In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.split_cp import SplitConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)
(
    proper_train_input_points,
    calib_input_points,
    proper_train_output_points,
    calib_output_points,
) = train_test_split(train_input_points, train_output_points, random_state=0)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs",
    loss_name=loss_name,
    loss_params=loss_params,
)
predictor.fit(proper_train_input_points, proper_train_output_points)

Instantiate region predictor

In [7]:
conformal_predictor = SplitConformalPredictor(predictor, non_conformity_name="absolute")
conformal_predictor.fit(calib_input_points, calib_output_points)
region_predictor = conformal_predictor.predict(test_input_points)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [9]:
coverage = np.mean(
    [
        test_output_point in prediction_region
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage)

test coverage:  0.848


In [10]:
prediction_regions

[[array([-1.57786156]),array([1.59099202])],
 [array([-1.58025339]),array([1.58860018])],
 [array([-1.5783134]),array([1.59054018])],
 [array([-1.57801399]),array([1.59083958])],
 [array([-1.57618539]),array([1.59266818])],
 [array([-1.57879205]),array([1.59006152])],
 [array([-1.57681479]),array([1.59203878])],
 [array([-1.57934733]),array([1.58950625])],
 [array([-1.57997988]),array([1.58887369])],
 [array([-1.58060086]),array([1.58825271])],
 [array([-1.57901088]),array([1.58984269])],
 [array([-1.57881615]),array([1.59003742])],
 [array([-1.57778012]),array([1.59107345])],
 [array([-1.57918676]),array([1.58966681])],
 [array([-1.57772364]),array([1.59112993])],
 [array([-1.57944722]),array([1.58940636])],
 [array([-1.57711081]),array([1.59174276])],
 [array([-1.577684]),array([1.59116958])],
 [array([-1.58032043]),array([1.58853314])],
 [array([-1.58136233]),array([1.58749124])],
 [array([-1.58056808]),array([1.5882855])],
 [array([-1.57719601]),array([1.59165757])],
 [array([-1.57